# Generate A Chirp Phase Table

Build a phase-only chirp waveform, visualize it, and write the phase column to a text file for TNMR sequence-table use.


## 1. Parameters

Edit these values for a different chirp.


In [ ]:
TABLE_NAME = "chirp1"

POINTS = 20000
DWELL_US = 0.2
SWEEP_WIDTH_HZ = -160.0e3


## 2. Setup


In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt

repo_search_start = Path.cwd().resolve()
REPO_ROOT = next(
    candidate
    for candidate in (repo_search_start, *repo_search_start.parents)
    if (candidate / "hardware_capture" / "rf" / "redstone" / "sequence.py").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REDSTONE_DIR = REPO_ROOT / "hardware_capture" / "rf" / "redstone"
OUTPUT_DIR = REDSTONE_DIR / "examples" / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TXT_PATH = OUTPUT_DIR / f"{TABLE_NAME}_phase_table.txt"

from hardware_capture.rf.redstone import Chirp

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)


## 3. Build The Chirp


In [ ]:
phase_waveform = Chirp(
    points=POINTS,
    sweep_width_hz=SWEEP_WIDTH_HZ,
    dwell_s=DWELL_US * 1.0e-6,
    offset_hz=-SWEEP_WIDTH_HZ / 2.0,
    phase=0.0,
    phase_steps=2**16,
)

duration_us = POINTS * DWELL_US
duration_ms = duration_us * 1.0e-3
frequency_hz = [value - phase_waveform.offset_hz for value in phase_waveform.frequency_values()]

{
    "table": TABLE_NAME,
    "points": POINTS,
    "dwell_us": DWELL_US,
    "duration_us": duration_us,
    "sweep_width_hz": SWEEP_WIDTH_HZ,
    "phase_mod_offset_hz": phase_waveform.offset_hz,
    "phase_mod_frequency_span_hz": (frequency_hz[0], frequency_hz[-1]),
    "waveform": phase_waveform.summary().to_dict(),
}


## 4. Visualize The Chirp Pulse


In [ ]:
time_ms = [index * phase_waveform.dwell_s * 1.0e3 for index in range(POINTS)]
quantized_phase_deg = phase_waveform.phase_values()
if quantized_phase_deg is None:
    raise ValueError("Phase output is not available.")

envelope_i = [math.cos(math.radians(value)) for value in quantized_phase_deg]
envelope_q = [math.sin(math.radians(value)) for value in quantized_phase_deg]
plot_step = max(1, POINTS // 5000)
plot_indices = list(range(0, POINTS, plot_step))
if plot_indices[-1] != POINTS - 1:
    plot_indices.append(POINTS - 1)

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot([time_ms[i] for i in plot_indices], [frequency_hz[i] * 1.0e-3 for i in plot_indices])
axes[0].set_ylabel("Frequency (kHz)")
axes[0].set_title(f"{TABLE_NAME}: {duration_ms:.3f} ms chirp")
axes[0].grid(True, alpha=0.3)

axes[1].plot([time_ms[i] for i in plot_indices], [quantized_phase_deg[i] for i in plot_indices])
axes[1].set_ylabel("Phase (deg)")
axes[1].set_ylim(-5, 365)
axes[1].grid(True, alpha=0.3)

axes[2].plot([time_ms[i] for i in plot_indices], [envelope_i[i] for i in plot_indices], label="I")
axes[2].plot([time_ms[i] for i in plot_indices], [envelope_q[i] for i in plot_indices], label="Q")
axes[2].set_xlabel("Time (ms)")
axes[2].set_ylabel("Unit envelope")
axes[2].grid(True, alpha=0.3)
axes[2].legend(loc="upper right")

fig.tight_layout()
plt.show()


## 5. Preview And Write The Phase Table


In [ ]:
phase_rows = phase_waveform.phase_column(precision=3)
if phase_rows is None:
    raise ValueError("Phase output is not available.")

output_path = phase_waveform.write_phase_column(OUTPUT_TXT_PATH, precision=3)

{
    "output_path": output_path,
    "row_count": len(phase_rows),
    "first_10": phase_rows[:10],
    "last_10": phase_rows[-10:],
}
